# MCP Tool Catalog

This notebook connects to the local `perseus` MCP server and displays every exposed tool, including its description and input schema.

> Requirements: run from the repository root (or keep the path setup cell unchanged) and install project dependencies with `uv sync` or `pip install -e .`.

In [1]:
from pathlib import Path
import importlib
import json
import sys

from IPython.display import Markdown, display

# Make `import server` work when the notebook is opened from examples/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from fastmcp import Client
import server

# Reload local edits when this notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp

## List all tools

`client.list_tools()` retrieves the same tool catalog that an external MCP-capable application sees.

In [2]:
async with Client(mcp) as client:
    tools = await client.list_tools()

print(f"Total tools: {len(tools)}")
for index, tool in enumerate(tools, start=1):
    summary = (tool.description or "No description").splitlines()[0]
    print(f"{index:>2}. {tool.name}: {summary}")

Total tools: 23
 1. get_passage: Get the text of a specific passage using a CTS URN.
 2. get_passage_plus: Get passage text plus surrounding metadata/context for a CTS URN.
 3. get_passage_plaintext: Get a passage as plain readable text instead of raw CTS XML.
 4. get_valid_references: Get valid citations/references for a work, useful for navigation.
 5. get_valid_references_json: Get valid citation references as paged JSON instead of raw CTS XML.
 6. count_valid_references: Count valid citation references without returning the full reference list.
 7. get_capabilities: Get the list of available texts and editions from Perseus CTS.
 8. get_cache_status: Get local metadata cache status.
 9. refresh_metadata_cache: Refresh cached CTS capabilities metadata from Perseus.
10. clear_metadata_cache: Clear local metadata cache files and in-memory cache entries.
11. list_text_groups: List authors/textgroups and their works from CTS capabilities.
12. get_author_resources: List CTS works/editions

## Descriptions and input schemas

The detailed catalog below shows each tool description and its JSON input schema.

In [3]:
for tool in tools:
    description = tool.description or "No description provided."
    schema = json.dumps(tool.inputSchema, ensure_ascii=False, indent=2)
    display(
        Markdown(
            f"## `{tool.name}`\n\n"
            f"{description}\n\n"
            f"**Input schema**\n\n"
            f"```json\n{schema}\n```"
        )
    )

## `get_passage`

Get the text of a specific passage using a CTS URN.

Examples:
- urn:cts:greekLit:tlg0012.tlg001:1.1-1.10
- urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_passage_plus`

Get passage text plus surrounding metadata/context for a CTS URN.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_passage_plaintext`

Get a passage as plain readable text instead of raw CTS XML.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_valid_references`

Get valid citations/references for a work, useful for navigation.

Optionally pass a citation `level` to constrain returned references.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    },
    "level": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_valid_references_json`

Get valid citation references as paged JSON instead of raw CTS XML.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    },
    "level": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    },
    "limit": {
      "default": 100,
      "type": "integer"
    },
    "offset": {
      "default": 0,
      "type": "integer"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

...

## `search_within_text`

Search within a single Scaife text/edition URN.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string"
    },
    "text_urn": {
      "type": "string"
    },
    "language": {
      "default": "greek",
      "type": "string"
    },
    "query_format": {
      "default": "auto",
      "type": "string"
    },
    "search_kind": {
      "default": "form",
      "type": "string"
    },
    "preserve_operators": {
      "default": false,
      "type": "boolean"
    },
    "size": {
      "default": 10,
      "type": "integer"
    },
    "offset": {
      "default": 0,
      "type": "integer"
    }
  },
  "required": [
    "query",
    "text_urn"
  ],
  "type": "object"
}
```

## `get_passage_highlights`

Get Scaife token highlight positions for a query within one passage.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string"
    },
    "passage_urn": {
      "type": "string"
    },
    "language": {
      "default": "greek",
      "type": "string"
    },
    "query_format": {
      "default": "auto",
      "type": "string"
    },
    "search_kind": {
      "default": "form",
      "type": "string"
    },
    "preserve_operators": {
      "default": false,
      "type": "boolean"
    }
  },
  "required": [
    "query",
    "passage_urn"
  ],
  "type": "object"
}
```

## `get_scaife_library_metadata`

Get Scaife JSON metadata for a textgroup, work, edition, or translation URN.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_scaife_passage_json`

Get Scaife JSON passage metadata/content for a passage URN.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## `get_scaife_passage_text`

Get Scaife plaintext for a passage URN.

**Input schema**

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

## Machine-readable catalog

Use this cell when you want the complete tool metadata as ordinary Python dictionaries.

In [4]:
tool_catalog = [tool.model_dump() for tool in tools]
print(json.dumps(tool_catalog, ensure_ascii=False, indent=2))

[
  {
    "name": "get_passage",
    "title": null,
    "description": "Get the text of a specific passage using a CTS URN.\n\nExamples:\n- urn:cts:greekLit:tlg0012.tlg001:1.1-1.10\n- urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "inputSchema": {
      "additionalProperties": false,
      "properties": {
        "urn": {
          "type": "string"
        }
      },
      "required": [
        "urn"
      ],
      "type": "object"
    },
    "outputSchema": {
      "properties": {
        "result": {
          "type": "string"
        }
      },
      "required": [
        "result"
      ],
      "type": "object",
      "x-fastmcp-wrap-result": true
    },
    "icons": null,
    "annotations": null,
    "meta": {
      "fastmcp": {
        "tags": []
      }
    },
    "execution": null
  },
  {
    "name": "get_passage_plus",
    "title": null,
    "description": "Get passage text plus surrounding metadata/context for a CTS URN.",
    "inputSchema": {
      "additionalPropert

# Notebook version

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.1</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 17, 2026</td>
    </tr>
  </table>
</div>